# Model Building and Hyperparameter Tuning

This notebook trains baseline models and performs hyperparameter tuning for a fraud detection classifier.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


In [3]:
# Load processed dataset
# Expected columns: category, amt, gender, state, city_pop, job, distance, trans_hour, trans_minute, trans_second, is_fraud
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "projects":
    project_root = project_root.parent

data_path = project_root / "data" / "processed_credit_data.csv"
data = pd.read_csv(data_path)

data.head()


,amt,category,gender,is_fraud,state,city_pop,job,distance_km,age,trans_hour,trans_dayofweek,trans_month
0,2.86,personal_care,M,0,SC,333497,Mechanical engineer,24.561462,52.257358,12,6,6
1,29.84,personal_care,F,0,UT,302,"Sales professional, IT",104.925092,30.425736,12,6,6
2,41.28,health_fitness,F,0,NY,34496,"Librarian, public",59.080078,49.667351,12,6,6
3,60.05,misc_pos,M,0,FL,54767,Set designer,27.698567,32.908966,12,6,6
4,3.19,travel,M,0,MI,1126,Furniture designer,104.335106,64.960986,12,6,6


In [5]:
target = "is_fraud"
X = data.drop(columns=[target])
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(categorical_cols)
numeric_cols = [col for col in X.columns if col not in categorical_cols]
print(numeric_cols)

print("Train class balance:", y_train.value_counts(normalize=True))
print("Test class balance:", y_test.value_counts(normalize=True))

['category', 'gender', 'state', 'job']
['amt', 'city_pop', 'distance_km', 'age', 'trans_hour', 'trans_dayofweek', 'trans_month']
Train class balance: is_fraud
0    0.99614
1    0.00386
Name: proportion, dtype: float64
Test class balance: is_fraud
0    0.99614
1    0.00386
Name: proportion, dtype: float64


In [8]:
# Preprocessing and baseline models
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop",
)

models = {
    "log_reg": Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]
    ),
    "random_forest": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    random_state=42,
                    n_jobs=-1,
                    class_weight="balanced",
                ),
            ),
        ]
    ),
    "gbdt": Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", GradientBoostingClassifier(random_state=42)),
        ]
    ),
}

def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test):
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    print("*****************************************")
    print(f"{name} metrics")
    print("ROC AUC:", round(roc_auc_score(y_test, proba), 4))
    print("Avg precision:", round(average_precision_score(y_test, proba), 4))
    print("Confusion matrix:", confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, digits=4))
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
    }

baseline_results = []
for name, pipe in models.items():
    baseline_results.append(evaluate_model(name, pipe, X_train, y_train, X_test, y_test))

*****************************************
log_reg metrics
ROC AUC: 0.9652
Avg precision: 0.1931
Confusion matrix: [[98560 12155]
 [   49   380]]
              precision    recall  f1-score   support

           0     0.9995    0.8902    0.9417    110715
           1     0.0303    0.8858    0.0586       429

    accuracy                         0.8902    111144
   macro avg     0.5149    0.8880    0.5002    111144
weighted avg     0.9958    0.8902    0.9383    111144

*****************************************
random_forest metrics
ROC AUC: 0.9991
Avg precision: 0.9208
Confusion matrix: [[110711      4]
 [   157    272]]
              precision    recall  f1-score   support

           0     0.9986    1.0000    0.9993    110715
           1     0.9855    0.6340    0.7716       429

    accuracy                         0.9986    111144
   macro avg     0.9920    0.8170    0.8855    111144
weighted avg     0.9985    0.9986    0.9984    111144

*****************************************
gbdt

In [9]:
# Hyperparameter tuning for Random Forest
rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1,
                class_weight="balanced",
            ),
        ),
    ]
)

param_dist = {
    "model__n_estimators": [200, 400, 600, 800],
    "model__max_depth": [None, 6, 10, 16, 24],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)

# Subsample for faster tuning
tune_size = 50000
if len(X_train) > tune_size:
    tune_idx = y_train.sample(n=tune_size, random_state=42).index
    X_tune = X_train.loc[tune_idx]
    y_tune = y_train.loc[tune_idx]
else:
    X_tune = X_train
    y_tune = y_train

search.fit(X_tune, y_tune)

best_model = search.best_estimator_
print("Best params:", search.best_params_)

_ = evaluate_model("tuned_random_forest", best_model, X_train, y_train, X_test, y_test)


Best params: {'model__n_estimators': 800, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': None}
*****************************************
tuned_random_forest metrics
ROC AUC: 0.9991
Avg precision: 0.9195
Confusion matrix: [[110715      0]
 [   183    246]]
              precision    recall  f1-score   support

           0     0.9983    1.0000    0.9992    110715
           1     1.0000    0.5734    0.7289       429

    accuracy                         0.9984    111144
   macro avg     0.9992    0.7867    0.8640    111144
weighted avg     0.9984    0.9984    0.9981    111144



In [10]:
# Save model
import joblib

model_path = Path("models/credit_risk_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, model_path)
model_path


PosixPath('models/credit_risk_model.pkl')

In [14]:
# Example inference
loaded_model = joblib.load(model_path)

new_data = pd.DataFrame(
    {
        "category": ["food_dining"],
        "amt": [25000.75],
        "gender": ["male"],
        "state": ["CA"],
        "city_pop": [100000],
        "job": ["engineer"],
        "distance_km": [5.0],
        "trans_hour": [12],
        "trans_dayofweek": [4],
        "trans_month": [5],
        "age": [30]
    }
)

new_proba = loaded_model.predict_proba(new_data)[:, 1]
new_pred = (new_proba >= 0.5).astype(int)
print("Predicted fraud probability:", round(float(new_proba[0]), 4))
print("Predicted class:", int(new_pred[0]))


Predicted fraud probability: 0.0175
Predicted class: 0


In [15]:
{
  "category": "food_dining",
  "amt": 150.75,
  "gender": "male",
  "state": "CA",
  "city_pop": 100000,
  "job": "engineer",
  "distance": 5.0,
  "trans_hour": 12,
  "trans_dayofweek": 3,
  "trans_month": 5,
  "age": 30
}

{'category': 'food_dining',
 'amt': 150.75,
 'gender': 'male',
 'state': 'CA',
 'city_pop': 100000,
 'job': 'engineer',
 'distance': 5.0,
 'trans_hour': 12,
 'trans_dayofweek': 3,
 'trans_month': 5,
 'age': 30}